In [1]:
# Quick demonstration: time the dense matvec at increasing num.
import time
import brainpy.math as bm
import numpy as np
from canns.models.basic import CANN1D

bm.set_dt(0.1)
results = []
for num in [128, 512, 2048, 4096]:
    m = CANN1D(num=num, accl_mode="normal")
    r = bm.random.rand(num)
    # Warm up
    for _ in range(3):
        m.irec_backend(r)
    t0 = time.perf_counter()
    for _ in range(30):
        m.irec_backend(r)
    results.append((num, (time.perf_counter() - t0) / 30 * 1e6))

print(f"{'num':>6}  {'dense (μs/step)':>16}")
for num, t in results:
    print(f"{num:>6}  {t:>16.1f}")


   num   dense (μs/step)
   128               7.6
   512               6.7
  2048              10.7
  4096              37.0


In [2]:
from canns.models.basic import CANN1D
import brainpy.math as bm

bm.set_dt(0.1)
m = CANN1D(num=128, accl_mode="fast")
print(f"  accl_mode:    {m.accl_mode}")
print(f"  accl_k:       {m.accl_k}")
print(f"  is_accelerated: {m.is_accelerated}")
print(f"  backend type: {type(m.irec_backend).__name__}")
print(f"  backend.mode: {m.irec_backend.mode}")
print(f"  backend.k:    {m.irec_backend.k}")
print(f"  U shape:      {m.irec_backend.U.shape}")
print(f"  V shape:      {m.irec_backend.V.shape}")


  accl_mode:    fast
  accl_k:       8
  is_accelerated: True
  backend type: LowRankIrec
  backend.mode: fast
  backend.k:    8
  U shape:      (128, 8)
  V shape:      (128, 8)


In [3]:
m = CANN1D(num=128, accl_mode="normal")
print(f"  type: {type(m.irec_backend).__name__}")
print(f"  k={m.accl_k}  (sentinel for full-rank / no approximation)")
print(f"  U is None: {m.irec_backend.U is None}")


  type: DenseIrec
  k=-1  (sentinel for full-rank / no approximation)
  U is None: True


In [4]:
m = CANN1D(num=2048, accl_mode="fast")
print(f"  type: {type(m.irec_backend).__name__}")
print(f"  k={m.accl_k}  (default for CANN1D fast)")
print(f"  factor shapes: U={m.irec_backend.U.shape}, V={m.irec_backend.V.shape}")


  type: LowRankIrec
  k=8  (default for CANN1D fast)
  factor shapes: U=(2048, 8), V=(2048, 8)


In [5]:
m = CANN1D(num=2048, accl_mode="ultra-fast")
print(f"  type: {type(m.irec_backend).__name__}")
print(f"  k={m.accl_k}  (ultra-fast = k=1 for CANN1D)")


  type: LowRankIrec
  k=1  (ultra-fast = k=1 for CANN1D)


In [6]:
m = CANN1D(num=2048, accl_mode="auto", accl_target_err_mrad=0.5)
print(f"  mode:    {m.accl_mode}")
print(f"  picked k={m.accl_k}  (for a 0.5 mrad budget)")

# Tightening the budget at runtime picks a higher k.
m.set_accl_mode("auto", target_err_mrad=0.1)
print(f"  after tightening to 0.1 mrad: k={m.accl_k}")


  mode:    auto
  picked k=5  (for a 0.5 mrad budget)


  after tightening to 0.1 mrad: k=8


In [7]:
# 1) Default grid (endpoint=True) — falls back to dense with a warning.
import warnings
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    m = CANN1D(num=128, accl_mode="fft")
print(f"  requested: fft → got: {m.accl_mode}  (silent fallback)")
print(f"  warning: {[str(w.message) for w in caught][0][:80]}...")


  requested: fft → got: normal  (silent fallback)


In [8]:
# 2) Clean grid (endpoint=False) — the FFT path becomes exact.
m = CANN1D(num=128, accl_mode="normal")
m.x = bm.linspace(-bm.pi, bm.pi, 128, endpoint=False)
m.conn_mat = m.make_conn()
m.set_accl_mode("fft")
print(f"  mode: {m.accl_mode}, K_fft shape: {m.irec_backend.K_fft.shape}")
print(f"  backend type: {type(m.irec_backend).__name__}")


  mode: fft, K_fft shape: (128,)
  backend type: CirculantFFTIrec1D


In [9]:
# Verify: the FFT matvec equals the dense matvec to ~1e-5 on the clean grid.
r = bm.random.rand(128)
dense = np.asarray(m.conn_mat @ r)
fft_out = np.asarray(m.irec_backend(r))
print(f"  max |dense - fft|: {np.abs(dense - fft_out).max():.2e}")


  max |dense - fft|: 1.14e-05


In [10]:
bm.set_dt(0.1)
num = 256
T = 80
m_n = CANN1D(num=num, accl_mode="normal")
m_f = CANN1D(num=num, accl_mode="fast")
x = np.asarray(m_n.x)
z_range = float(m_n.z_range)

def run(m):
    traj = np.empty((T, num), dtype=np.float32)
    for t in range(T):
        pos = np.pi * t / (T - 1)
        d = (x - pos) % z_range
        d = np.where(d > z_range / 2, d - z_range, d)
        inp = (m.A * np.exp(-0.25 * (d / m.a) ** 2)).astype(np.float32)
        m.update(inp)
        traj[t] = np.asarray(m.r.value)
    return traj

r_n, r_f = run(m_n), run(m_f)
print(f"  r_max trajectory max diff: {np.abs(r_n.max(1) - r_f.max(1)).max():.2e}")
print(f"  ✓ (the fast and dense models agree to ~1e-7 over 80 steps)")


  r_max trajectory max diff: 9.53e-07
  ✓ (the fast and dense models agree to ~1e-7 over 80 steps)


In [11]:
bm.set_dt(0.1)
m = CANN1D(num=256, accl_mode="normal")
print(f"  start:       mode={m.accl_mode}  backend={type(m.irec_backend).__name__}")
m.set_accl_mode("fast")
print(f"  → fast:      mode={m.accl_mode}  backend={type(m.irec_backend).__name__}  k={m.accl_k}")
m.set_accl_mode("auto", target_err_mrad=0.5)
print(f"  → auto:      mode={m.accl_mode}  backend={type(m.irec_backend).__name__}  k={m.accl_k}")
m.set_accl_mode("normal")
print(f"  → normal:    mode={m.accl_mode}  backend={type(m.irec_backend).__name__}")


  start:       mode=normal  backend=DenseIrec
  → fast:      mode=fast  backend=LowRankIrec  k=8
  → auto:      mode=auto  backend=LowRankIrec  k=5
  → normal:    mode=normal  backend=DenseIrec
